# Part d: L1 and L2 regularization


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax

from src.data import make_runge_split, make_runge_split_3, design_matrix, scale_data, center_data, runge
from src.functions import mse, mse_der, sigmoid, sigmoid_der, identity, identity_der
from src.network import NeuralNetwork
from src.optimizers import Adam
from src.regression import ols_fit, ridge_fit, lasso_fit
from src.metrics import r2
from src.plotting import line_plot, heatmap, save_fig

Dataset and best result from project 1: 
-n = 100 points
-sigma = 0.1 noise
-seed = 2026
-test size = 0.2
-Optimization method = Adam

In [ ]:
# recreating ridge and lasso results from project 1
# Use the same dataset and the best results from project 1.
x_train, x_test, y_train, y_test = make_runge_split()                   # split 80/20
x_trn, x_val, x_tst, y_trn, y_val, y_tst = make_runge_split_3()         # 64/16/20, for tuning
assert np.array_equal(x_tst, x_test) and np.array_equal(y_tst, y_test)  # same test set in both splits

print("train", x_train.shape, " test", x_test.shape)
print("tuning: train", x_trn.shape, " validation", x_val.shape)

degree = 8                      # best OLS and Ridge
degree_lasso = 5                # best Lasso          
num_iters = 20000 
lam_ols = 0.0
lam_ridge = 1e-4/len(x_train)   # best Ridge
lam_lasso = 1e-2                # used for Lasso
gamma = 0.01
seed = 2026

In [ ]:
# design matrix
def prepare(deg):
    matrix_train = design_matrix(x_train, deg, intercept=False)
    matrix_test = design_matrix(x_test, deg, intercept=False)
    P_train, P_test, P_mean, P_std = scale_data(matrix_train, matrix_test)
    
    def poly_pred(theta, x):
        P = (design_matrix(x, deg, intercept=False) - P_mean) / P_std
        return P @ theta + y_mean
    return P_train, P_test, poly_pred

P8_train, P8_test, predict8 = prepare(8)     # OLS and Ridge
P5_train, P5_test, predict5 = prepare(5)     # Lasso
y_train_c, y_test_c, y_mean = center_data(y_train, y_test)


# regression fits
theta_ols = ols_fit(P8_train, y_train_c)
theta_ridge = ridge_fit(P8_train, y_train_c, lam_ridge)
theta_lasso = lasso_fit(P5_train, y_train_c, lam_lasso, gamma = 0.01, n_iter = 20000)

# predictions
results = {}
for name, theta, predict in [(f'OLS   - deg = {degree}', theta_ols, predict8),
                    (f'Ridge - deg = {degree}, lam = {lam_ridge}', theta_ridge, predict8),
                    (f'Lasso - deg = {degree_lasso}, lam = {lam_lasso}', theta_lasso, predict5)]:
    
    y_pred = predict(theta, x_test)
    MSE = mse(y_pred, y_test)
    R2 = r2(y_pred, y_test)
    results[name] = (float(MSE), R2)
    print(f'{name:32s} | test MSE {results[name][0]:.4f} | R2 {results[name][1]:.3f}')

### Using 1D Runge function

In [ ]:
X_trn_s, X_val_s, x_mean_trn, x_std_trn = scale_data(x_trn, x_val) # scale with 64 tuning-training points
epochs = 500
batch_size = 14

def make_net(lam, reg):
    # input - hidden layers - linear output, MSE cost, Adam, (not penalising biases)
    return NeuralNetwork(network_input_size = 1,
                        layer_output_sizes = [50 ,1],
                        activation_funcs = [sigmoid, identity], 
                        activation_ders = [sigmoid_der, identity_der],
                        cost_fun = mse,
                        cost_der = mse_der,
                        optimizer = Adam(gamma),
                        regularization = reg,
                        lmbda = lam,
                        seed = seed,
                        penalize_bias=False)


def train_net(X_fit, y_fit, X_eval, y_eval, reg, lam):
    # training the network on the fits and returning it + MSE on the validations
    neural_net = make_net(lam, reg)
    neural_net.train(X_fit, y_fit, epochs=epochs, batch_size=batch_size, rng = np.random.default_rng(seed))
    MSE = mse(neural_net.predict(X_eval), y_eval)
    score = float(MSE)
    return neural_net, (score if np.isfinite(score) else np.nan)


def gradient_check(network, X, y): 
    # compare compute_gradient with jax.grad
    def cost_layers(layers, X, y):
        network.layers = layers
        return network.cost(X, y, include_penalty=True)
    
    net_layers = network.layers
    grad = network.compute_gradient(X, y)
    grad_jax = jax.grad(cost_layers)(net_layers, X, y) # put the real weights back

    network.layers = net_layers
    return [(bool(np.allclose(dW, dW_jax)), bool(np.allclose(db, db_jax))) for (dW, db), (dW_jax, db_jax) in zip(grad, grad_jax)]



### Add hyperparameters $\lambda$ with the $L_2$ and $L_1$ norms (sec 8.12).

In [ ]:
# Adding L1-Lasso to network
L1_net = make_net(lam = lam_lasso, reg = 'L1')
print('L1 gradient check against jax, (dW, db) per layer:', gradient_check(L1_net, X_trn_s, y_trn))

for l in [0.0, 1e-4, 1e-2]:
    val = train_net(X_trn_s, y_trn, X_val_s, y_val, reg = 'L1', lam = l)[1]
    print(f'L1: lam = {l:6g}: validation MSE {val:.4f}')

In [ ]:
# Adding L2-Ridge to network
L2_net = make_net(reg = 'L2', lam = lam_ridge)
print('L2 gradient check against jax, (dW, db) per layer:', gradient_check(L2_net, X_trn_s, y_trn))

for l in [0.0, 1e-4, 1e-2]:
    val = train_net(X_trn_s, y_trn, X_val_s, y_val, reg = 'L2', lam = l)[1]
    print(f'L2: lam = {l:6g}: validation MSE {val:.4f}')

In [ ]:
# Find the optimal results for the hyperparameters lambda, learning rates gamma and neural network architecture.
lam_vals = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
gamma_vals = [1e-3, 3e-3, 1e-2, 3e-2, 1e-1]

val_mse, best = {}, {}
for reg in ['L2', 'L1']:
    Z = np.full((len(lam_vals), len(gamma_vals)), np.nan)
    for i, lam in enumerate(lam_vals):
        for j, gamma in enumerate(gamma_vals):
            Z[i,j] = train_net(X_trn_s, y_trn, X_val_s, y_val, reg, lam)[1]
    
    Z[Z > 1] = np.nan
    val_mse[reg] = Z
    i, j = np.unravel_index(np.nanargmin(Z), Z.shape)
    best[reg] = dict(lam = lam_vals[i], gamma = gamma_vals[j], val = Z[i, j])

    heatmap(Z, gamma_vals, lam_vals, r'learning rate $\gamma$', r'$\lambda$ ('+ reg + ')', label = 'validation MSE', name = f'd_heatmap_{reg.lower()}')
    plt.show()

    print(f'{reg}: best lambda = {lam_vals[i]:g}, gamma = {gamma_vals[j]:g}, validation MSE {Z[i, j]:.4f}')


In [ ]:
# Compare the $L_2$ results with Ridge regression from project 1.
# Use the same dataset and the best results from project 1.

In [ ]:
# Compare the $L_1$ results with the Lasso calculations of project 1.
# Use the same dataset and the best results from project 1.